# Lesson 31: Embeddings and vector databases — FAISS, ChromaDB, Qdrant, Pinecone 🗄️
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Create **sentence embeddings** and compare meanings with cosine similarity
2. Understand what a **vector database** does: store, index, search, filter
3. Use **FAISS**, **ChromaDB** and **Qdrant** (and see how **Pinecone** works in the cloud)
4. Measure why **approximate** search is needed at scale

**⏱️ Timing:** embeddings 15' | FAISS 15' | Chroma & Qdrant 20' | Pinecone & comparison 10'

> 💡 Runs on CPU; a GPU makes embedding faster.

## 🗺️ An analogy: a library organised by meaning
A normal database is like a library sorted **alphabetically** — great if you know the exact title.
A **vector database** is a library sorted by **meaning**: books about similar ideas sit on nearby shelves, so "a book like this one" is a quick walk away.

- **Embedding** = a list of numbers representing the meaning of a text, image or sound (Lesson 22 did this for single words; today, whole sentences)
- **Vector database** = stores millions of embeddings and finds the **nearest neighbours** to a query very fast, often with extra **metadata** filters ("only Year 9 documents")

In [ ]:
!pip -q install sentence-transformers faiss-cpu chromadb qdrant-client
import numpy as np, time, warnings; warnings.filterwarnings("ignore")
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")      # small, fast, 384 numbers per text

sentences = ["Phones must be kept in lockers during class.",
             "Can I use my mobile in maths?",
             "The canteen sells sushi on Tuesdays.",
             "What food is available at lunch?",
             "The library closes at 4:30pm.",
             "Dogs are loyal pets."]
E = embedder.encode(sentences, normalize_embeddings=True)
print("Each sentence →", E.shape[1], "numbers. First 6 for sentence 1:", E[0][:6].round(3))

In [ ]:
import matplotlib.pyplot as plt
sim = E @ E.T     # with normalised vectors, dot product = cosine similarity
plt.figure(figsize=(7, 6)); plt.imshow(sim, cmap="Blues", vmin=0, vmax=1)
short = [s[:28] + "…" for s in sentences]
plt.xticks(range(len(short)), short, rotation=60, ha="right"); plt.yticks(range(len(short)), short)
for i in range(len(sim)):
    for j in range(len(sim)): plt.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("cosine similarity of meanings"); plt.show()

Notice *"Phones must be kept in lockers"* and *"Can I use my mobile in maths?"* are similar **even though they share almost no words**. TF-IDF (Lessons 21 and 29) couldn't do that.

### 🧩 Exercise 1
Add 4 sentences: two that mean the same with different words, and two that share words but mean different things ("I saw a bat in the cave" / "He swung the bat"). Does the heatmap agree with you?

## Part 2: A dataset to search 📚
We'll embed 640 short science facts (made up from templates so the notebook works offline).

In [ ]:
import random
random.seed(0)
subjects = {"biology": ["cells", "the heart", "plants", "DNA", "bacteria", "the brain", "bones", "muscles"],
            "physics": ["gravity", "magnets", "light", "sound", "electricity", "friction", "energy", "heat"],
            "chemistry": ["acids", "atoms", "metals", "water", "salt", "oxygen", "carbon", "reactions"],
            "earth science": ["volcanoes", "earthquakes", "glaciers", "rivers", "clouds", "rocks", "oceans", "soil"]}
templates = ["Year {y} students learn how {t} work in {s} class.", "A Year {y} fact sheet about {t} from {s}.",
             "In {s}, the topic of {t} is introduced in Year {y}.", "Experiments with {t} are popular in Year {y} {s}.",
             "Revision notes on {t} for Year {y} {s}."]
docs, meta = [], []
for s, topics in subjects.items():                       # every combination once → no duplicates
    for t in topics:
        for y in [7, 8, 9, 10]:
            for tpl in templates:
                docs.append(tpl.format(y=y, t=t, s=s)); meta.append({"subject": s, "year": y})
start = time.time()
doc_vecs = embedder.encode(docs, normalize_embeddings=True, batch_size=64)
print(f"Embedded {len(docs)} docs in {time.time() - start:.1f}s →", doc_vecs.shape)

## Part 3: FAISS — fast search from Meta AI ⚡
**FAISS** (Facebook AI Similarity Search) is a **library**, not a server: it keeps vectors in memory and searches them extremely fast. Used inside many larger systems.

In [ ]:
import faiss
index = faiss.IndexFlatIP(doc_vecs.shape[1])      # "Flat" = compare with every vector (exact); IP = inner product
index.add(doc_vecs)
q = embedder.encode(["how does the human body pump blood"], normalize_embeddings=True)
scores, ids = index.search(q, 3)
for s, i in zip(scores[0], ids[0]):
    print(f"{s:.2f}  {docs[i]}")

In [ ]:
# Why "approximate" search? Let's time exact search on 200,000 vectors vs an approximate index
# real embeddings form clusters of similar meanings, so we fake 200,000 vectors grouped around 2,000 "topics"
centres = np.random.randn(2000, 384).astype("float32")
big = centres[np.random.randint(0, 2000, 200_000)] + 0.5 * np.random.randn(200_000, 384).astype("float32")
faiss.normalize_L2(big)
query = big[:100]

flat = faiss.IndexFlatIP(384); flat.add(big)
t = time.time(); _, exact = flat.search(query, 5); t_flat = time.time() - t

quantizer = faiss.IndexFlatIP(384)
ivf = faiss.IndexIVFFlat(quantizer, 384, 256, faiss.METRIC_INNER_PRODUCT)   # split space into 256 "neighbourhoods"
ivf.train(big[:20_000]); ivf.add(big); ivf.nprobe = 8                       # only search the 8 nearest neighbourhoods
t = time.time(); _, approx = ivf.search(query, 5); t_ivf = time.time() - t

recall = np.mean([len(set(a) & set(e)) / 5 for a, e in zip(approx, exact)])
print(f"Exact search:       {t_flat:.3f}s")
print(f"Approximate search: {t_ivf:.3f}s  ({t_flat / t_ivf:.0f}× faster), finds {recall:.0%} of the true top-5")

### 🧩 Exercise 2
Change `ivf.nprobe` to 1, 32 and 128. How do **speed** and **recall** trade off? (Real vector databases make this same trade-off with clever indexes like **HNSW** graphs.)

## Part 4: ChromaDB — a simple vector database 🎨
**Chroma** is an open-source vector database that's easy to start with: it stores the text, the vectors **and** metadata, and can filter.

In [ ]:
import chromadb
chroma = chromadb.Client()                                       # in memory; chromadb.PersistentClient("folder") saves to disk
col = chroma.create_collection("science_facts", metadata={"hnsw:space": "cosine"})
col.add(ids=[str(i) for i in range(len(docs))], documents=docs, embeddings=doc_vecs.tolist(), metadatas=meta)

def chroma_search(question, where=None, k=3):
    r = col.query(query_embeddings=embedder.encode([question], normalize_embeddings=True).tolist(), n_results=k, where=where)
    for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0]):
        print(f"{1 - dist:.2f}  {m}  {d}")

chroma_search("lava and eruptions")
print("\nOnly Year 7:")
chroma_search("lava and eruptions", where={"year": 7})

## Part 5: Qdrant — a production vector database 🚀
**Qdrant** is an open-source vector database written in Rust, built to run as a server for big apps. For learning, it can also run **inside Python in memory**.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, Filter, FieldCondition, MatchValue

qdrant = QdrantClient(":memory:")       # for a real server: QdrantClient(url="http://localhost:6333")
qdrant.create_collection("science_facts", vectors_config=VectorParams(size=384, distance=Distance.COSINE))
qdrant.upsert("science_facts", points=[PointStruct(id=i, vector=doc_vecs[i].tolist(), payload={"text": docs[i], **meta[i]})
                                        for i in range(len(docs))])

def qdrant_search(question, subject=None, k=3):
    flt = Filter(must=[FieldCondition(key="subject", match=MatchValue(value=subject))]) if subject else None
    hits = qdrant.query_points("science_facts", query=embedder.encode(question, normalize_embeddings=True).tolist(),
                               query_filter=flt, limit=k).points
    for h in hits:
        print(f"{h.score:.2f}  {h.payload['subject']:14} {h.payload['text']}")

qdrant_search("experiments with magnets")
print("\nOnly chemistry:")
qdrant_search("experiments with magnets", subject="chemistry")

## Part 6: Pinecone — a managed cloud vector database ☁️
**Pinecone** runs entirely in the cloud: no server to manage, you just send vectors with an API key (free starter tier; account holders must be adults — teacher demo). The cell skips itself without a key.

In [ ]:
try:
    from google.colab import userdata
    PINECONE_KEY = userdata.get("PINECONE_API_KEY")
except Exception:
    PINECONE_KEY = None
    print("⏭️  No PINECONE_API_KEY in Colab Secrets — skipping the Pinecone demo.")

if PINECONE_KEY:
    !pip -q install pinecone
    from pinecone import Pinecone, ServerlessSpec
    pc = Pinecone(api_key=PINECONE_KEY)
    if "science-facts" not in pc.list_indexes().names():
        pc.create_index("science-facts", dimension=384, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    idx = pc.Index("science-facts")
    idx.upsert(vectors=[(str(i), doc_vecs[i].tolist(), {"text": docs[i], **meta[i]}) for i in range(200)])
    r = idx.query(vector=embedder.encode("volcano", normalize_embeddings=True).tolist(), top_k=3, include_metadata=True)
    for m in r.matches: print(f"{m.score:.2f}  {m.metadata['text']}")

## ⚖️ Which one should I use?
| | FAISS | ChromaDB | Qdrant | Pinecone |
|---|---|---|---|---|
| Type | Library | Database (embedded or server) | Database server | Managed cloud service |
| Stores text & metadata? | ❌ vectors only | ✅ | ✅ | ✅ |
| Filtering | ❌ (DIY) | ✅ | ✅ (very rich) | ✅ |
| Best for | Raw speed, research, custom systems | Prototypes, small apps | Self-hosted production | Production without running servers |
| Cost | Free | Free (open source) | Free (open source) / cloud plans | Paid beyond the free tier |

### 🚀 Challenge
Embed the 30 most recent notices from your school website (copy and paste), store them in Chroma with a `date` field, and build a search that only returns notices from this month.

## 📌 Summary
- **Embeddings** capture meaning; **cosine similarity** compares them
- **Vector databases** store embeddings and find nearest neighbours fast, with **metadata filters**
- **Exact** search compares everything; **approximate** indexes (IVF, HNSW) trade a little accuracy for big speed-ups
- FAISS (library) · Chroma (easy DB) · Qdrant (production DB) · Pinecone (managed cloud)